# React — Deployment

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> Two lessons, and they are the difference between an app that works on your machine and one that
> works on the internet. Everything measured here was measured against the course playground on
> Vite 8.3.0.

## LESSON 91 — The production build

Since LESSON 2 you have run `npm run dev`. That command is a development server: it transforms
each module as the browser asks for it, and it is not what you deploy.

### Two commands

```bash
npm run build      # writes dist/ — the files you actually deploy
npm run preview    # serves dist/ locally, so you can check it before deploying
```

`preview` matters more than it looks. It is the only way to see the *built* app before it is
public: minified, bundled, with `NODE_ENV=production`, without StrictMode's double-invocation and
without the development warnings. Bugs that appear only in production usually appear here first.

Building the playground prints its inventory:

```text
dist/index.html                   0.40 kB │ gzip:  0.27 kB
dist/assets/index-BCoMbhzZ.css    0.58 kB │ gzip:  0.36 kB
dist/assets/index-C2NvOFWp.js   220.48 kB │ gzip: 68.93 kB
✓ built in 114ms
```

Three things worth reading in that output. The **hash in each filename** (`index-C2NvOFWp.js`)
changes when the contents change, which is what lets a host cache the file forever and still serve
you a new one after a deploy. The **gzip column** is what the user actually downloads — 68.93 kB,
not 220. And `dist/` is **static files**: no Node process runs in production, which is why hosting
one of these costs nothing.

### `import.meta.env`

Vite exposes environment information on `import.meta.env`. Measured in the playground, in
development:

```text
MODE: development
DEV: true   PROD: false
BASE_URL: /
```

`MODE` is `"production"` and the two booleans flip in a build. Use them for things that should
differ, like verbose logging:

```js
if (import.meta.env.DEV) {
  console.log("[debug]", state);
}
```

Your own variables go in a `.env` file at the project root, and here is the rule that matters:

```bash
VITE_API_URL=https://api.example.com
SECRET_TOKEN=super-secret-value
```

Only names starting with **`VITE_`** are exposed to your code. Measured, in the browser:

```text
VITE_GREETING: hello-from-env
SECRET_TOKEN:                     ← undefined
```

### `VITE_*` variables are not secrets

This is the most important sentence in the lesson, and it is measurable rather than a matter of
opinion. Building with both variables and searching the output bundle:

```text
is "super-secret-value" in dist/assets/*.js ?   0 matches
is "hello-from-env"     in dist/assets/*.js ?   1 match
```

The `VITE_` value is **inlined into the JavaScript, in plain text**, because it must be — the code
runs in the user's browser, and the browser has to have the value. Anyone can read it with View
Source.

So: **prefixing a secret with `VITE_` does not protect it; it publishes it.** An API key that
grants write access, a database URL, a private token — none of them can live in a front-end build,
whatever you name them. They belong on a server that your front end calls.

What legitimately goes in a `VITE_` variable: your API's public base URL, a public analytics id, a
feature flag, the app's version. Things that are already public, or harmless if read.

And `.env` goes in `.gitignore` — not because the values are secret, but because they differ per
environment. Commit a `.env.example` listing the names with empty values, so the next person knows
what to set.

### Key Notes

- `npm run build` writes static files to `dist/`; `npm run preview` serves them so you can check
  before deploying.
- Hashed filenames make aggressive caching safe; the gzip column is the real download size.
- `import.meta.env.DEV` / `.PROD` / `.MODE` for behaviour that should differ; only `VITE_*`
  variables reach your code.
- A `VITE_` variable is compiled into the bundle in plain text. It is public. Secrets belong on a
  server.

### Example

**Runnable — plain JS.** The one part of this lesson that is logic rather than commands: deciding
what may go in a front-end environment variable. Getting this wrong is the most common real
security mistake made by people who have just learned to deploy.

In [ ]:
// L91 — may this go in the bundle?

const l91Candidates = [
  { name: "VITE_API_URL", value: "https://api.example.com", note: "the public base URL of our API" },
  { name: "VITE_STRIPE_PUBLISHABLE_KEY", value: "pk_live_123", note: "designed to be public" },
  { name: "VITE_STRIPE_SECRET_KEY", value: "sk_live_123", note: "grants charges" },
  { name: "VITE_DATABASE_URL", value: "postgres://user:pw@host/db", note: "direct database access" },
  { name: "VITE_APP_VERSION", value: "1.4.2", note: "shown in the footer" },
  { name: "VITE_ADMIN_EMAIL", value: "ada@example.com", note: "a real person's address" },
  { name: "API_SIGNING_SECRET", value: "hunter2", note: "no VITE_ prefix" },
];

function l91Check({ name, value, note }) {
  const exposed = name.startsWith("VITE_");

  const looksSecret =
    /secret|private|password|signing/i.test(name) ||
    /^sk_|^postgres:\/\/|^mysql:\/\//.test(value) ||
    /:\/\/[^/]*:[^/@]*@/.test(value);                 // credentials inside a URL

  const personal = /email|phone/i.test(name);

  if (!exposed) return { verdict: "not exposed", why: "no VITE_ prefix — your code cannot read it either" };
  if (looksSecret) return { verdict: "DANGER", why: "this would be published in plain text in the bundle" };
  if (personal) return { verdict: "think again", why: "not a secret, but personal data in a public file" };
  return { verdict: "fine", why: note };
}

for (const candidate of l91Candidates) {
  const { verdict, why } = l91Check(candidate);
  console.log(`${verdict.padEnd(13)} ${candidate.name.padEnd(30)} ${why}`);
}

console.log(
  "\nThe test is not 'is it prefixed' but 'am I willing for a stranger to read this'.",
  "\nThe prefix decides whether your CODE can see it. The build decides that the WORLD can.",
);

### Exercise

Part 1 is **runnable — plain JS**; parts 2 and 3 are **in the playground**.

1. `l91Check` misses things. Add detection for a JWT (three base64 segments separated by dots), an
   AWS-style key (`AKIA` followed by capitals and digits), and any value over 100 characters that
   is not a URL — long opaque strings are usually credentials. Test it against three fake values
   you invent, and then say in a comment why a check like this can only ever be a safety net.
2. **In the playground:** run `npm run build`, then `npm run preview`, and open the URL it prints.
   Compare it with `npm run dev`: does the console still show React's double-invocation from
   StrictMode? Explain the difference in one sentence.
3. Add a `.env` with one `VITE_` variable and one without, log both from a component, and build.
   Search `dist/assets/*.js` for both values and record what you find. Then delete the `.env` and
   the log — this is a measurement, not a change to keep.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** Where does each of these belong: the bundle, a server, or neither?

```
1. the URL of your public REST API
2. the key that lets you send email through a provider
3. a feature flag deciding whether a beta panel shows
4. the logged-in user's session token
5. your Google Analytics measurement id
6. the connection string for the production database
7. the app's git commit hash, shown in the footer
8. an admin password used by a nightly job
```

Write `l91Where(item)` returning `"bundle"`, `"server"` or `"neither"` with a reason. Then answer
in comments:

1. Number 4 is interesting — a session token is genuinely secret, yet it lives in the browser.
   Why is that different from putting it in the bundle?
2. Two of these are things people routinely and wrongly put in a front-end `.env`. Which two, and
   what would you tell a colleague who says "but the repository is private"?

In [ ]:
// Your code here

## LESSON 92 — Deploying a single-page app

The last lesson of the course, and the one that saves you an evening of confusion the first time
you deploy something with a router.

### What you are deploying

`dist/` — an `index.html`, some hashed JavaScript and CSS, and your static assets. No server code,
no Node process. Any static host can serve it, and the three obvious ones are free for this.

| host | how |
|---|---|
| **Vercel** | connect the repository; it detects Vite, builds, and deploys on every push |
| **Netlify** | the same, with build command `npm run build` and publish directory `dist` |
| **GitHub Pages** | a workflow that builds and publishes `dist/` — free with the repo you already have |

All three redeploy when you push. Set your `VITE_` variables in the host's dashboard, not in a
committed file.

### The SPA fallback — the one thing that will catch you

Your app has routes (topic 20). `/projects/7` works perfectly in development. You deploy, click
through to it, and it works. Then you press **reload**, and you get a **404**.

Here is why, and it is not a React problem. Clicking a `<Link>` never contacts the server —
React Router changes the URL in the browser and renders a different component (LESSON 58). But a
reload *does* contact the server, asking for `/projects/7`. The server looks for a file at that
path, finds only `index.html` and `assets/`, and returns 404. It has never heard of your routes,
because your routes only exist inside the JavaScript it was about to send.

The fix is one rule, called the **SPA fallback**: *for any path that is not a real file, serve
`index.html`.* The app then boots, React Router reads the URL, and renders the right route.

**Netlify** — `public/_redirects`, copied into `dist/` by the build:

```text
/*    /index.html   200
```

**Vercel** — `vercel.json` at the project root:

```json
{ "rewrites": [{ "source": "/(.*)", "destination": "/index.html" }] }
```

**GitHub Pages** has no rewrite rules at all. The standard workaround is to copy `index.html` to
`404.html` in `dist/`, so the "not found" page *is* the app.

The `200` in the Netlify rule matters: it is a rewrite, not a redirect. The URL the user typed
stays in the address bar, which is the whole point — a redirect to `/` would lose the route.

**How to test it before you tell anyone.** Deploy, navigate to a deep route by clicking, then
press reload. If it survives a reload and a pasted link in a new tab, the fallback is right.

### The base path

If the app is not at the root of the domain — GitHub Pages serves at
`https://you.github.io/repo-name/` — every asset URL must include that prefix.

```bash
npm run build -- --base=/react-project-management/
```

Measured, that changes `dist/index.html` from `src="/assets/index-*.js"` to:

```html
src="/react-project-management/assets/index-CU6PSa4d.js"
```

Get it wrong and you see the symptom everyone sees once: **a blank page, and 404s for the JS and
CSS in the Network tab.** The HTML loaded and asked for `/assets/…`, which does not exist at the
root of that domain.

Two more things need the same prefix. `import.meta.env.BASE_URL` holds it, so use that rather than
hard-coding paths to files in `public/`. And React Router needs to know:

```jsx
<BrowserRouter basename={import.meta.env.BASE_URL}>
```

Vercel and Netlify serve at a domain root, so the base stays `/` and none of this applies. It is a
GitHub Pages concern, and a sub-path concern generally.

### A deployment checklist

1. `npm run build` succeeds with no warnings you have not read.
2. `npm run preview` — the built app works, not just the dev server.
3. The SPA fallback is configured for your host.
4. The base path is set if you are not at a domain root.
5. `VITE_` variables are set in the host's dashboard; nothing secret is among them (LESSON 91).
6. `.env` is gitignored; `.env.example` is committed.
7. Deploy, then **reload a deep route** and **paste a deep link into a new tab**.
8. Put the live URL in the repository's README and its About field.

### Key Notes

- You deploy static files; any static host will do, and all three suggested ones are free.
- Without an SPA fallback, reloading a deep route 404s — the server has never heard of your routes.
- Netlify `_redirects`, Vercel `rewrites`, GitHub Pages a `404.html` copy.
- Not at a domain root? Set `--base`, use `import.meta.env.BASE_URL`, and pass `basename` to the
  router.

### Example

**Runnable — plain JS.** The fallback rule is a routing decision, and writing it makes the 404
obvious. This is what the host does, in six lines.

In [ ]:
// L92 — what a static host does with a request

const l92Files = new Set([
  "/index.html",
  "/assets/index-CU6PSa4d.js",
  "/assets/index-BCoMbhzZ.css",
  "/favicon.svg",
]);

// a plain static server: a file, or 404
function l92Plain(path) {
  const candidate = path === "/" ? "/index.html" : path;
  return l92Files.has(candidate) ? { status: 200, served: candidate } : { status: 404, served: null };
}

// the same server with an SPA fallback
function l92WithFallback(path) {
  const direct = l92Plain(path);
  if (direct.status === 200) return direct;
  return { status: 200, served: "/index.html", rewritten: true };
}

const l92Requests = ["/", "/index.html", "/assets/index-CU6PSa4d.js", "/projects", "/projects/7", "/favicon.svg"];

console.log("path                          plain          with fallback");
for (const path of l92Requests) {
  const plain = l92Plain(path);
  const fallback = l92WithFallback(path);
  console.log(
    `${path.padEnd(30)}${String(plain.status).padEnd(15)}${fallback.status}${fallback.rewritten ? "  (rewritten to /index.html)" : ""}`,
  );
}

// the request that breaks an unconfigured deploy:
console.log("\nreloading /projects/7 without a fallback:", l92Plain("/projects/7").status);
console.log("with a fallback:", l92WithFallback("/projects/7").status, "→", l92WithFallback("/projects/7").served);
console.log("\nNote what the fallback does NOT do: it never invents a file. It serves index.html and");
console.log("lets the router — which is inside that JavaScript — decide what /projects/7 means.");

### Exercise

Part 1 is **runnable — plain JS**; parts 2 and 3 are **in your project**.

1. The fallback above rewrites *everything* that is not a file, including `/api/tasks` and a
   mistyped `/asssets/main.js`. Write `l92Smarter(path)` that returns a real 404 for anything
   under `/api/` or `/assets/`, and falls back only for paths that could plausibly be a route.
   Then say in a comment why serving `index.html` for a missing asset is actively harmful — think
   about what the browser does with HTML it was expecting to be JavaScript.
2. **In your project:** build with `--base=/some-path/` and open `dist/index.html` in an editor.
   Record the `src` and `href` values. Then open that file directly in a browser and describe what
   you see and why.
3. Deploy something. Any of the three hosts, any of your mini-projects — Mini-project 1 has no
   router and is the easiest first deploy. Then deploy one *with* a router, break the fallback on
   purpose, reload a deep route to see the 404, and fix it. You will only make that mistake once,
   and it is much cheaper to make deliberately.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** Deployment debugging, from symptom to cause. This is the lookup table you
will actually want in six months.

For each symptom, name the most likely cause and the fix:

```
1. a blank page, and 404s for /assets/*.js in the Network tab
2. the home page works; reloading /projects/7 gives 404
3. the home page works; /projects/7 redirects to / and loses the route
4. the app works, but every API call fails with a URL like "undefined/tasks"
5. it works locally with npm run dev, and the deployed build shows a blank page with a console
   error about a bare specifier
6. an old version keeps loading after a deploy, until a hard refresh
```

Write `l92Diagnose(symptom)` returning `{ cause, fix }`, then answer in comments:

1. Which of the six would `npm run preview` have caught before deploying, and which would it not?
2. Number 6 is about caching. Given that Vite puts a content hash in every asset filename, which
   single file is the one that must **not** be cached aggressively — and why does that one file
   make the whole scheme work?

In [ ]:
// Your code here

> **LESSON 92 — the end of the course.**
>
> Ninety-two lessons ago you wrote a function that returned a string, and the point being made was
> that a UI can be a function of state. Everything since has been that idea with more of the
> details filled in: JSX, props, state, effects, refs, reducers, context, routing, custom hooks,
> architecture, performance, error handling, forms and actions, testing, a store, and now
> deployment.
>
> What is left is Mini-project 4, which uses most of it at once — and, after that, the only thing
> that actually makes any of this stick: building something nobody set as an exercise.